In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

In [ ]:
dataset_path = "datasets/shoeDataset"

X = []
y = []

# The dataset has two sub-folders (training, validation), each with one folder per class.
# We read both and make our own 70/15/15 split below.
class_names = sorted(os.listdir(os.path.join(dataset_path, "training")))

print("Classes:", class_names)

for part in ["training", "validation"]:

    for i, class_name in enumerate(class_names):

        folder_path = os.path.join(dataset_path, part, class_name)

        for image_name in os.listdir(folder_path):

            image_path = os.path.join(folder_path, image_name)

            try:
                img = Image.open(image_path).convert("RGB")

                img = img.resize((150, 150))

                img = np.array(img, dtype=np.float32) / 255.0

                X.append(img)
                y.append(i)

            except Exception:
                print("Skipping:", image_path)


X = np.array(X, dtype=np.float32)
y = np.array(y)

print("X shape:", X.shape)
print("y shape:", y.shape)


In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)

In [ ]:
X_train = torch.tensor(X_train, dtype=torch.float32)
X_val = torch.tensor(X_val, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.long)
y_val = torch.tensor(y_val, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)


# Change image dimensions from NHWC to NCHW

X_train = X_train.permute(0, 3, 1, 2)
X_val = X_val.permute(0, 3, 1, 2)
X_test = X_test.permute(0, 3, 1, 2)


print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

train_data = TensorDataset(X_train, y_train)
val_data = TensorDataset(X_val, y_val)
test_data = TensorDataset(X_test, y_test)

train_loader = DataLoader(
    train_data,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_data,
    batch_size=32,
    shuffle=False
)

test_loader = DataLoader(
    test_data,
    batch_size=32,
    shuffle=False
)

In [ ]:
class CNN(nn.Module):

    def __init__(self, num_classes):

        super(CNN, self).__init__()

        self.conv1 = nn.Conv2d(3, 32, kernel_size=3)
        self.pool = nn.MaxPool2d(kernel_size=2) 

        self.conv2 = nn.Conv2d(32, 64, kernel_size=3)

        self.conv3 = nn.Conv2d(64, 128, kernel_size=3)

        self.flatten = nn.Flatten()

        self.fc1 = nn.Linear(128 * 17 * 17, 128)

        self.dropout = nn.Dropout(0.5)

        self.fc2 = nn.Linear(128, num_classes)


    def forward(self, x):

        x = self.conv1(x)
        x = torch.relu(x)
        x = self.pool(x)

        x = self.conv2(x)
        x = torch.relu(x)
        x = self.pool(x)

        x = self.conv3(x)
        x = torch.relu(x)
        x = self.pool(x)

        x = self.flatten(x)

        x = self.fc1(x)
        x = torch.relu(x)

        x = self.dropout(x)

        x = self.fc2(x)

        return x

In [ ]:
num_classes = len(class_names)

model = CNN(num_classes)

print(model)

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=0.0001
)

In [ ]:
epochs = 30

train_losses = []
val_losses = []

train_accuracies = []
val_accuracies = []


for epoch in range(epochs):

    # Training mode
    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:

        # Forward propagation
        outputs = model(images)

        loss = criterion(outputs, labels)

        # Backpropagation
        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item() * images.size(0)

        # Predictions
        predictions = torch.argmax(outputs, dim=1)

        correct += (predictions == labels).sum().item()

        total += labels.size(0)


    train_loss = total_loss / total
    train_acc = correct / total


    # Validation
    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            outputs = model(images)

            loss = criterion(outputs, labels)

            total_loss += loss.item() * images.size(0)

            predictions = torch.argmax(outputs, dim=1)

            correct += (predictions == labels).sum().item()

            total += labels.size(0)


    val_loss = total_loss / total
    val_acc = correct / total


    # Store results
    train_losses.append(train_loss)
    val_losses.append(val_loss)

    train_accuracies.append(train_acc)
    val_accuracies.append(val_acc)


    print(
        f"Epoch [{epoch+1}/{epochs}] "
        f"Train Loss: {train_loss:.4f} "
        f"Train Acc: {train_acc*100:.2f}% "
        f"Val Loss: {val_loss:.4f} "
        f"Val Acc: {val_acc*100:.2f}%"
    )

In [ ]:
epochs_range = range(1, len(train_losses) + 1)

fig, ax1 = plt.subplots(figsize=(10, 6))

# Loss
ax1.plot(epochs_range, train_losses, label="Training Loss")
ax1.plot(epochs_range, val_losses, label="Validation Loss")

ax1.set_xlabel("Epochs")
ax1.set_ylabel("Loss")

# Accuracy
ax2 = ax1.twinx()

ax2.plot(
    epochs_range,
    train_accuracies,
    '--',
    label="Training Accuracy"
)

ax2.plot(
    epochs_range,
    val_accuracies,
    '--',
    label="Validation Accuracy"
)

ax2.set_ylabel("Accuracy")

# Combine legends
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()

ax1.legend(
    lines1 + lines2,
    labels1 + labels2,
    loc="center right"
)

plt.title("Training vs Validation Performance")

plt.show()